In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             precision_recall_curve, roc_curve, f1_score, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Data Exploration

## Data Kit01

In [ ]:
import pandas as pd
import os
import re

def load_Monorail(filepath):
    df = pd.read_csv(filepath)
    df = df[df['Non_Standard_Braking'] == 0]

    # Extract numeric part from filename and convert to integer
    match = re.search(r'kit(\d+)', os.path.basename(filepath))
    source = int(match.group(1)) if match else -1  # fallback to -1 if no match
    df['Source'] = source

    for col in df.select_dtypes(include='object'):
        try:
            df[col] = df[col].str.replace(' sec', '', regex=False).astype(float)
        except ValueError:
            continue
    return df


# List of file paths
filepaths = [
    'TestBrakefinal_data_kit01.csv',
    'TestBrakefinal_data_kit06.csv',
    'TestBrakefinal_data_kit27.csv'
]

# Process all files
dfs = [load_Monorail(fp) for fp in filepaths]

# Combine into one DataFrame
df_data = pd.concat(dfs, ignore_index=True)

print(df_data[['Source']].value_counts())
print(df_data.shape)

### Data kit01 - Fills missing NaN with median value

In [ ]:
# from sklearn.impute import SimpleImputer
# import pandas as pd

# # Assuming df_data is already defined as in your code
# imputer = SimpleImputer(strategy='median')

# # Apply imputer to numeric columns only
# df_data[df_data.select_dtypes(include='number').columns] = imputer.fit_transform(df_data.select_dtypes(include='number'))

# # df_filtered = df_data[(df_data['WV_MeanPressure'] >= 2) & (df_data['WV_MeanPressure'] <= 3)]
# # print(df_filtered.shape)
# # df_filtered.head()

## San Donato Data

In [ ]:
df_reference = pd.read_csv('model.csv')
df_reference['Malfunction'] = df_reference['Malfunction'].astype(str)

# Create binary label
leakage_codes = ['C', 'D', 'E', 'F', 'G']
df_reference['LeakageLabel'] = np.where(df_reference['Malfunction'].isin(leakage_codes), 'Combined leakage', 'Healthy')

# Aggregate delay and efficiency columns
delay_eff_map = {
    'Total_timing_delay': ['Brake_timing_delay_exp', 'Release_timing_delay_exp'],
    'Total_energy_delay': ['Brake_energy_delay_exp', 'Release_energy_delay_exp'],
    'Total_power_delay': ['Brake_power_delay_exp', 'Release_power_delay_exp'],
    'Total_power_efficiency': ['Brake_power_efficiency_exp', 'Release_power_efficiency_exp'],
    'Total_energy_efficiency': ['Brake_energy_effiency_exp', 'Release_energy_efficiency_exp']
}

for new_col, sources in delay_eff_map.items():
    df_reference[new_col] = df_reference[sources[0]] + df_reference[sources[1]]

# Drop original columns
cols_to_drop = [col for pair in delay_eff_map.values() for col in pair]
df_reference.drop(columns=cols_to_drop, inplace=True)

# Rename columns
rename_map = {
    'Release_start_pressure_delay_exp': 'Release_start_pressure_delay',
    'Buildup_end_pressure_delay_exp': 'Buildup_end_pressure_delay',
    'Weight': 'WV_MeanPressure',
    'Brake_action': 'EmergencyBrake_action'
}
df_reference.rename(columns=rename_map, inplace=True)

### Feature Engineering

Split labeled data to features and target 

In [ ]:
common_cols = df_reference.columns.intersection(df_data.columns)

Features = df_reference[common_cols]
Features = Features.drop(columns=['LeakageLabel','WV_MeanPressure','EmergencyBrake_action'])
Parameters = df_reference[['WV_MeanPressure','EmergencyBrake_action','Brake_mode','Frequency','Sensor']]
Target = df_reference['LeakageLabel']
Target_raw = df_reference['Malfunction']

print(Features.shape)
Features.head()

Fills missing with median value instead of removing since we have little data

In [ ]:
# Impute missing (with median)
imp = SimpleImputer(strategy="median")
Features = pd.DataFrame(imp.fit_transform(Features), columns=Features.columns, index=Features.index)
#Features = Features.dropna(axis=1)

# Drop columns that are all NaN or constant
Features = Features.loc[:, Features.notna().any()]  # drop all-NaN
const_mask = Features.nunique(dropna=True) <= 1
if const_mask.any():
    Features = Features.loc[:, ~const_mask]

Features.shape
Features.columns

### Exploration - Feature Importance

In [ ]:
# === Unified Feature Selection Pipeline: RF, MI, ANOVA
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import mutual_info_classif, f_classif, RFE, RFECV
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
from sklearn.feature_selection import VarianceThreshold

Xsel = Features.copy()

# Keep only numeric columns (if any non-numeric slipped in)
num_cols = [c for c in Xsel.columns if np.issubdtype(Xsel[c].dtype, np.number)]
Xsel = Xsel[num_cols].copy()

# Example: drop features with variance below 1e-2 AFTER scaling (optional):
vt = VarianceThreshold(threshold=1e-2)
X_var = vt.fit_transform(Xsel)
kept_mask = vt.get_support()
kept_features = Xsel.columns[kept_mask]
Xsel = Xsel[kept_features]


# Some selectors need scaling
sc_std = StandardScaler()
sc_rob = RobustScaler()
X_std = pd.DataFrame(sc_std.fit_transform(Xsel), columns=Xsel.columns, index=Xsel.index)
X_rob = pd.DataFrame(sc_rob.fit_transform(Xsel), columns=Xsel.columns, index=Xsel.index)

# if y is string, change to 0/1
y_enc = pd.Series(Target).astype("category")
if y_enc.dtype.name == "category":
    y_enc = y_enc.cat.codes  # e.g., Leakage=1, Normal=0

# For stability on tiny datasets
cv = StratifiedKFold(n_splits=min(5, max(2, np.bincount(y_enc).min())), shuffle=True, random_state=42)

# Helper to convert scores to ranks (lower rank = better)
def to_rank(series, higher_is_better=True):
    s = series.copy()
    if not higher_is_better:
        s = -s
    # rank 1 = best
    return s.rank(ascending=False, method="average")

# ---------- 1) RandomForest importance ----------
# Measures a feature's utility in improving the model's prediction accuracy (e.g., mean decrease in impurity).
# Captures feature interactions naturally; highly effective.
# REDUCED trees for small dataset stability
rf = RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=3, 
                            random_state=42, class_weight="balanced")
rf.fit(Xsel, y_enc)
rf_imp = pd.Series(rf.feature_importances_, index=Xsel.columns, name="RF_Importance")
rf_rank = to_rank(rf_imp, higher_is_better=True).rename("RF_Rank")

# ---------- 2) Mutual Information ----------
# Measures statistical dependency (information gain) between a feature and the target.
# Captures non-linear relationships. Evaluates each feature independently; ignores feature interactions.
mi = mutual_info_classif(Xsel, y_enc, random_state=42, discrete_features=False, n_neighbors=3)
mi_score = pd.Series(mi, index=Xsel.columns, name="MI_Score")
mi_rank = to_rank(mi_score, higher_is_better=True).rename("MI_Rank")

# ---------- 3) ANOVA F-test ----------
# (works best if roughly Gaussian/scaled; we used imputed data)
# Measures linear correlation between a feature and the target by comparing variance between groups to variance within groups.
# Assumes linear relationship and Gaussian distribution; ignores feature interactions.
F_vals, p_vals = f_classif(Xsel, y_enc)
f_score = pd.Series(F_vals, index=Xsel.columns, name="ANOVA_F")
f_rank = to_rank(f_score, higher_is_better=True).rename("ANOVA_Rank")

# ---------- Combine all rankings with OPTIMIZED WEIGHTS for 60 samples ----------
rank_table = pd.concat([rf_rank, mi_rank, f_rank,
                        rf_imp, mi_score, f_score], axis=1)

# WEIGHTED OverallRank for small datasets (60 samples)
# MI: 0.50 (highest weight - most reliable for small data)
# ANOVA: 0.30 (second - stable if linear relationships exist)
# RF: 0.20 (lowest - prone to overfitting with 60 samples)
weights = {
    "MI_Rank": 0.70,
    "ANOVA_Rank": 1,
    "RF_Rank": 0.20
}

rank_table["OverallRank"] = (
    rank_table["MI_Rank"] * weights["MI_Rank"] +
    rank_table["ANOVA_Rank"] * weights["ANOVA_Rank"] +
    rank_table["RF_Rank"] * weights["RF_Rank"]
)

# Sort and display top-N
N = 10
rank_table_sorted = rank_table.sort_values("OverallRank").head(N)
print("=== Top features by Weighted OverallRank (lower = better) ===")
print(f"Weights: MI={weights['MI_Rank']}, ANOVA={weights['ANOVA_Rank']}, RF={weights['RF_Rank']}")
display(rank_table_sorted)

plt.figure(figsize=(8, max(4, 0.35*N)))
rank_table_sorted.sort_values("OverallRank")["OverallRank"].plot(kind="barh")
plt.gca().invert_yaxis()
plt.title(f"Top {N} Features by (Weight more on Anova)")
plt.xlabel("Rank (lower is better)")
plt.tight_layout()
plt.show()

topN_features = rank_table_sorted.index.tolist()
print("\nTopN feature list:", topN_features)

Features_reduced = Features[topN_features]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =============================================================
# CORRELATION PRUNING USING OVERALL RANK (lower = better)
# =============================================================

corr = Features_reduced.corr().abs()

# mask for upper triangle
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

threshold = 0.90
to_drop = []
clusters = []  # for reporting

for col in upper.columns:
    # features correlated with "col"
    correlated = list(upper.index[upper[col] > threshold])

    if len(correlated) > 0:
        group = [col] + correlated
        group = list(set(group))

        # record the group
        clusters.append(group)

        # choose the best feature based on OverallRank
        best_feature = (
            rank_table.loc[group]["OverallRank"]
            .sort_values()
            .index[0]
        )

        # others must be dropped
        drop_these = [f for f in group if f != best_feature]
        to_drop.extend(drop_these)

# make unique
to_drop = list(set(to_drop))

# =============================================================
# APPLY DROPPING
# =============================================================
final_features = [f for f in Features_reduced.columns if f not in to_drop]
X_final = Features_reduced[final_features]


# =============================================================
# PRINT REPORT
# =============================================================
print("=== Correlation Groups (|rho| > 0.90) ===")
for g in clusters:
    print(f"Group: {g}")
    best = rank_table.loc[g]["OverallRank"].sort_values().index[0]
    print(f"→ Keeping: {best}")
    print()

print("\n=== Features DROPPED due to high correlation ===")
print(to_drop)

print("\n=== FINAL FEATURE LIST ===")
print(final_features)


# =============================================================
# PLOT FINAL FEATURES
# =============================================================
plt.figure(figsize=(8, max(3, 0.35 * len(final_features))))
plt.barh(final_features, [1]*len(final_features))
plt.title("Final Selected Features After Correlation Pruning")
plt.xlabel("Kept (1 = yes)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.heatmap(Features_reduced.corr(), annot=True, cmap='coolwarm')
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
Feature_use = Features_reduced[final_features]
# sns.heatmap(Feature_use.corr(), annot=True, cmap='coolwarm')
# plt.title("Feature Correlation Heatmap")
# plt.show()

corr_matrix = Feature_use.corr().abs()

# Compute mean correlation for each feature
mean_corr = corr_matrix.mean().sort_values(ascending=False)

top5_features = mean_corr.head(5).index.tolist()
print("Top 5 features by average correlation:", top5_features)

sns.heatmap(
    Feature_use[top5_features].corr(),
    annot=True,
    cmap='coolwarm'
)
plt.title("Top 5 Feature Correlation Heatmap")
plt.show()

In [ ]:
def box_target_plotter(data, target):
    for col in data.select_dtypes("number"):
        sns.boxplot(data=data, x=target, y=col)
        plt.show()

box_target_plotter(Feature_use, Target)

## Combine Reference data with San Donato data

In [ ]:
# Combine common columns
common_cols = df_reference.columns.intersection(df_data.columns)

# Add DataSource column to each DataFrame
df_reference_subset = df_reference[common_cols].copy()
df_reference_subset['DataSource'] = 0

df_data_subset = df_data[common_cols].copy()
df_data_subset['DataSource'] = 1

# Concatenate the two DataFrames
df_combined = pd.concat([df_reference_subset, df_data_subset], ignore_index=True)

# Final label encoding
df_combined.rename(columns={'LeakageLabel': 'label'}, inplace=True)
df_combined['label'] = df_combined['label'].map({'Healthy': 0, 'Combined leakage': 1})

# Convert ' sec' strings to float
for col in df_combined.select_dtypes(include='object'):
    try:
        df_combined[col] = df_combined[col].str.replace(' sec', '', regex=False).astype(float)
    except ValueError:
        continue

df = df_combined.copy()

Target_combined = df['label']
print(df.shape)
df.head()

Intersect of Data Reference with San Donato Data to compare features

In [ ]:
df_filtered = df_data[common_cols]
df_filtered = df_filtered.drop(columns=['LeakageLabel'])
Target_data = df_data['LeakageLabel']

df_filtered.head()

### Combined Leakage Boxchart of Features 

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ---- inputs ----
# df: your combined dataframe (already built above)
# Features_reduced: list of column names you want to plot
# Target_combined = df['label']  # already defined

# ---- tidy data for plotting ----
plot_cols = ['label', 'DataSource'] + list(Features_reduced)
D = df[plot_cols].copy()

# Optional: restore human-readable class labels
label_map = {0: 'Healthy', 1: 'Leakage'}
D['label'] = D['label'].map(label_map).astype('category')

# Optional: name your data sources
source_map = {0: 'Reference', 1: 'Kit-01'}   # edit names if you like
D['DataSource'] = D['DataSource'].map(source_map).astype('category')

# Clean unusual numeric values
D.replace([np.inf, -np.inf], np.nan, inplace=True)

# Melt to long form: one row per (sample, feature)
D_long = D.melt(
    id_vars=['label','DataSource'],
    value_vars=Features_reduced,
    var_name='Feature', value_name='Value'
)
D_long = D_long.dropna(subset=['Value'])

g = sns.catplot(
    data=D_long, x='label', y='Value', hue='DataSource',
    col='Feature', col_wrap=3, kind='box', height=4, sharey=False
)
g.set_axis_labels("Class label", "Value")
g.set_titles("{col_name}")

Plot the Kit01 Data grouped by WV Mean Pressure

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- 1) Filter to DataSource == 1 ---
D = df.loc[df['DataSource'] == 1, ['label','WV_MeanPressure'] + list(Features_reduced)].copy()

# Optional: readable class labels for x-axis
label_map = {0: 'Healthy', 1: 'Leakage'}
D['label'] = D['label'].map(label_map).astype('category')

# --- 2) Bin WV_MeanPressure into [<2, 2–3, >3] ---
edges  = [-np.inf, 2.0, 3.0, np.inf]
labels = ['< 2', '2–3', '> 3']
D['WV_bin'] = pd.cut(D['WV_MeanPressure'], bins=edges, labels=labels, right=True, include_lowest=True)

# Clean up impossible values
D.replace([np.inf, -np.inf], np.nan, inplace=True)

# --- 3) Melt to long form: one row per (sample, feature) ---
D_long = D.melt(
    id_vars=['label','WV_bin'],
    value_vars=Features_reduced,
    var_name='Feature', value_name='Value'
).dropna(subset=['Value','WV_bin'])

# --- 4) Draw grouped boxplots (hue by WV_bin) ---
sns.set(style="whitegrid")
g = sns.catplot(
    data=D_long, x='label', y='Value', hue='WV_bin',
    col='Feature', col_wrap=3, kind='box', height=4, sharey=False
)
g.set_axis_labels("Class label", "Value")
g.set_titles("{col_name}")
g._legend.set_title("WV Mean Pressure (bar)")
plt.show()


Plot Healthy only subset of Kit 01 and Reference

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ==============================================================
# 1) Prepare "Healthy only" subset
# ==============================================================
cols_needed = ['label', 'DataSource', 'WV_MeanPressure'] + list(Features_reduced)
D = df[cols_needed].copy()

label_map   = {0: 'Healthy', 1: 'Leakage'}
source_map  = {0: 'Reference', 1: 'Kit-01'}
D['label']       = D['label'].map(label_map).astype('category')
D['DataSource']  = D['DataSource'].map(source_map).astype('category')

# Keep only Healthy samples
D = D[D['label'] == 'Healthy']

# ==============================================================
# 2) Bin WV_MeanPressure into [<2, 2–3, >3]
# ==============================================================
edges  = [-np.inf, 2.0, 3.0, np.inf]
labels = ['< 2', '2–3', '> 3']
D['WV_bin'] = pd.cut(
    D['WV_MeanPressure'], bins=edges, labels=labels,
    right=True, include_lowest=True
)

# ==============================================================
# 3) Loop over each feature and make a separate figure
# ==============================================================
sns.set(style="whitegrid")

for feat in Features_reduced:
    plt.figure(figsize=(6, 5), dpi=300)  # increase size + resolution
    ax = sns.boxplot(
        data=D,
        x='DataSource', y=feat,
        hue='WV_bin',
        order=['Reference', 'Kit-01'],
        hue_order=['< 2', '2–3', '> 3']
    )
    ax.set_title(f"{feat} — Healthy only", fontsize=13)
    ax.set_xlabel("Data Source", fontsize=11)
    ax.set_ylabel("Value", fontsize=11)
    ax.legend(title="WV Mean Pressure (bar)")
    plt.tight_layout()
    plt.show()
    # Optionally save each plot:
    # plt.savefig(f"{feat}_Healthy_boxplot.png", dpi=300, bbox_inches='tight')


### COMBINED LEAKAGE - Variation Error of Features

In [ ]:
import numpy as np
import pandas as pd

# ---------------- safety: make sure Features_reduced is a list of strings ----------------
if isinstance(Features_reduced, (str, bytes)):
    Features_reduced = [Features_reduced]
else:
    Features_reduced = list(Features_reduced)

# Keep only columns we need (and that actually exist)
base_cols = ['label', 'DataSource', 'WV_MeanPressure']
use_cols  = [c for c in Features_reduced if c in df.columns]
missing   = sorted(set(Features_reduced) - set(use_cols))
if missing:
    print("Warning: missing features (skipped):", missing)

D = df[base_cols + use_cols].copy()

# Map labels and sources (works whether df has 0/1 or already strings)
label_map  = {0: 'Healthy', 1: 'Leakage'}
source_map = {0: 'Reference', 1: 'Kit-01'}
D['label'] = D['label'].map(label_map).fillna(D['label'])
D['DataSource'] = D['DataSource'].map(source_map).fillna(D['DataSource'])

# Healthy only, WV in [2, 3]
D = D[(D['label'] == 'Healthy') &
      (D['WV_MeanPressure'] >= 2.0) &
      (D['WV_MeanPressure'] <= 3.0)].copy()

# Coerce features to numeric (avoid weird objects like "1.5 sec")
for c in use_cols:
    D[c] = pd.to_numeric(D[c], errors='coerce')
D.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop rows where ALL features are NaN (keeps rows if at least one feature is present)
D = D.dropna(subset=use_cols, how='all')

# Median per DataSource, features as rows
median_table = (
    D.groupby('DataSource')[use_cols]
      .median()
      .T
    # Ensure both columns exist; if a source is missing, you'll get NaN
    .reindex(columns=['Reference', 'Kit-01'])
)

# Compute variation: |M1 - M0| / |M0| * 100
ref = median_table['Reference']
real = median_table['Kit-01']

# avoid division by zero warnings
den = ref.replace(0, np.nan)
median_table['Variation_%'] = (real.sub(ref).abs().div(den.abs()).mul(100))

# Optional: sort by largest variation
median_table = median_table.sort_values('Variation_%', ascending=True)

print("=== Median comparison (Healthy; WV 2–3 bar) ===")
print(median_table.round(3))

median_table.round(3).to_csv('Median_Comparison_Healthy_WV2-3bar.csv', index=True)


### MANUAL BRAKE ACTIVATION - Boxchart

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

T_firstphase = df_healthy.copy()
# 10 Hz features to plot
vars_10hz = [
    'First_phase_half_time_ratio',
    'First_phase_mean_curvature',
    'First_phase_power'
]

# WV_MeanPressure bins: [<2, 2–3, >3]
bins = [-np.inf, 2, 3, np.inf]
bin_labels = ['<2', '2–3', '>3']

# --- Loop over each BC_ID present in T_firstphase ---
for bc_id in T_firstphase['BC_ID'].unique():
    df_sensor = T_firstphase[T_firstphase['BC_ID'] == bc_id].copy()
    if df_sensor.empty:
        continue

    # All rows are already Standard braking
    df_sensor['brake_type'] = 'Service'

    # WV_MeanPressure bins
    wv = pd.to_numeric(df_sensor['WV_MeanPressure'], errors='coerce')
    df_sensor['wv_bin'] = pd.cut(
        wv,
        bins=bins,
        labels=bin_labels,
        right=True,        # (a,b]
        include_lowest=True
    )

    sensor_id = str(bc_id)

    # --- Figure for this sensor: 1x3 subplots ---
    fig, axes = plt.subplots(1, len(vars_10hz), figsize=(5*len(vars_10hz), 4), sharey=False)
    axes = np.atleast_1d(axes)

    for ax, var in zip(axes, vars_10hz):
        y10 = pd.to_numeric(df_sensor[var], errors='coerce')

        df_feat = pd.DataFrame({
            'value': y10,
            'brake_type': df_sensor['brake_type'],
            'wv_bin': df_sensor['wv_bin']
        }).dropna(subset=['value', 'wv_bin'])

        if df_feat.empty:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.axis('off')
            continue

        sns.boxplot(
            data=df_feat,
            x='brake_type',      # only "Standard"
            y='value',
            hue='wv_bin',        # WV_MeanPressure bins
            ax=ax
        )

        feat_name = var.replace('_', ' ')
        ax.set_title(f'{feat_name} — Sensor {sensor_id}')
        ax.set_xlabel('Braking type')
        ax.set_ylabel('Value')
        ax.grid(True, axis='y', linestyle='--', alpha=0.4)

    # Single legend for the whole figure
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, title='WV MeanPressure bin', loc='upper right')

    # Remove legends from individual subplots
    for ax in axes:
        if ax.get_legend() is not None:
            ax.get_legend().remove()

    fig.suptitle(f'First-phase (10 Hz) vs WV bins — Sensor {sensor_id}', y=1.05)
    fig.tight_layout()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

T_firstphase = df_healthy.copy()

# 10 Hz features to plot
vars_10hz = [
    'First_phase_half_time_ratio',
    'First_phase_mean_curvature',
    'First_phase_power'
]

# WV_MeanPressure bins: [<2, 2–3, >3]
bins = [-np.inf, 2, 3, np.inf]
bin_labels = ['<2', '2–3', '>3']

# --- Loop over each BC_ID present in T_firstphase ---
for bc_id in T_firstphase['BC_ID'].unique():
    df_sensor = T_firstphase[T_firstphase['BC_ID'] == bc_id].copy()
    if df_sensor.empty:
        continue

    # WV_MeanPressure bins
    wv = pd.to_numeric(df_sensor['WV_MeanPressure'], errors='coerce')
    df_sensor['wv_bin'] = pd.cut(
        wv,
        bins=bins,
        labels=bin_labels,
        right=True,
        include_lowest=True
    )

    sensor_id = str(bc_id)
    # Map EmergencyBrake_action to descriptive labels
    df_sensor['brake_label'] = df_sensor['EmergencyBrake_action'].map({0: 'Service', 1: 'Emergency'})
    # --- Figure for this sensor: 1x3 subplots ---
    fig, axes = plt.subplots(1, len(vars_10hz), figsize=(5*len(vars_10hz), 4), sharey=False)
    axes = np.atleast_1d(axes)

    for ax, var in zip(axes, vars_10hz):
        y10 = pd.to_numeric(df_sensor[var], errors='coerce')

        df_feat = pd.DataFrame({
            'value': y10,
            'brake_label': df_sensor['brake_label'],
            'wv_bin': df_sensor['wv_bin']
        }).dropna(subset=['value', 'wv_bin', 'brake_label'])


        if df_feat.empty:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.axis('off')
            continue

        sns.boxplot(
            data=df_feat,
            x='brake_label',
            y='value',
            hue='wv_bin',
            ax=ax
        )


        feat_name = var.replace('_', ' ')
        ax.set_title(f'{feat_name} — Sensor {sensor_id}')
        ax.set_xlabel('Brake Type')
        ax.set_ylabel('Value')
        ax.grid(True, axis='y', linestyle='--', alpha=0.4)

    # Single legend for the whole figure
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, title='WV MeanPressure bin', loc='upper right')

    # Remove legends from individual subplots
    for ax in axes:
        if ax.get_legend() is not None:
            ax.get_legend().remove()

    fig.suptitle(f'First-phase (10 Hz) vs WV bins — Sensor {sensor_id}', y=1.05)
    fig.tight_layout()

In [ ]:
T_firstphase = df_healthy.copy()
vars_10hz = [
    'First_phase_half_time_ratio',
    'First_phase_mean_curvature',
    'First_phase_power'
]

for bc_id in T_firstphase['BC_ID'].unique():
    df_sensor = T_firstphase[T_firstphase['BC_ID'] == bc_id].copy()
    if df_sensor.empty:
        continue

    # Map EmergencyBrake_action to readable labels
    df_sensor['brake_label'] = 'service'
    sensor_id = str(bc_id)

    fig, axes = plt.subplots(1, len(vars_10hz), figsize=(5*len(vars_10hz), 4), sharey=False)
    axes = np.atleast_1d(axes)

    for ax, var in zip(axes, vars_10hz):
        y10 = pd.to_numeric(df_sensor[var], errors='coerce')

        df_feat = pd.DataFrame({
            'value': y10,
            'brake_label': df_sensor['brake_label']
        }).dropna(subset=['value', 'brake_label'])

        if df_feat.empty:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.axis('off')
            continue

        sns.boxplot(
            data=df_feat,
            x='brake_label',
            y='value',
            ax=ax
        )

        feat_name = var.replace('_', ' ')
        ax.set_title(f'{feat_name} — Sensor {sensor_id}')
        ax.set_xlabel('Brake Type')
        ax.set_ylabel('Value')
        ax.grid(True, axis='y', linestyle='--', alpha=0.4)

    fig.suptitle(f'First-phase (10 Hz) — Sensor {sensor_id}', y=1.05)
    fig.tight_layout()

In [ ]:
# Select only sensor 0x94 from your already-filtered T_firstphase
df94 = T_firstphase[T_firstphase['BC_ID'] == '0x94'].copy()

# Features to summarize
vars_10hz = [
    'First_phase_half_time_ratio',
    'First_phase_mean_curvature',
    'First_phase_power'
]

# Convert to numeric and compute median
median_dict = {}
for var in vars_10hz:
    median_dict[var] = pd.to_numeric(df94[var], errors='coerce').median()

# Create a table (DataFrame)
median_table = pd.DataFrame.from_dict(median_dict, orient='index', columns=['Median'])
median_table.index.name = 'Feature'

print(median_table)


# Algorithm Combined

In [ ]:
# === Unified Feature Selection Pipeline: RF, MI, ANOVA
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import mutual_info_classif, f_classif, RFE, RFECV
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt

# Separate features and labels
Features = df.drop('label', axis=1)
Target = df['label']
Xsel = Features.copy()

# Impute missing (with median)
imp = SimpleImputer(strategy="median")
Features = pd.DataFrame(imp.fit_transform(Features), columns=Features.columns, index=Features.index)
#Features = Features.dropna(axis=1)
# Drop columns that are all NaN or constant
Features = Features.loc[:, Features.notna().any()]  # drop all-NaN
const_mask = Features.nunique(dropna=True) <= 1
if const_mask.any():
    Features = Features.loc[:, ~const_mask]

Xsel = Features.copy()
# Keep only numeric columns (if any non-numeric slipped in)
num_cols = [c for c in Xsel.columns if np.issubdtype(Xsel[c].dtype, np.number)]
Xsel = Xsel[num_cols].copy()

# Some selectors need scaling
sc_std = StandardScaler()
sc_rob = RobustScaler()
X_std = pd.DataFrame(sc_std.fit_transform(Xsel), columns=Xsel.columns, index=Xsel.index)
X_rob = pd.DataFrame(sc_rob.fit_transform(Xsel), columns=Xsel.columns, index=Xsel.index)

# if y is string, change to 0/1
y_enc = pd.Series(Target).astype("category")
if y_enc.dtype.name == "category":
    y_enc = y_enc.cat.codes  # e.g., Leakage=1, Normal=0

# For stability on tiny datasets
cv = StratifiedKFold(n_splits=min(5, max(2, np.bincount(y_enc).min())), shuffle=True, random_state=42)

# Helper to convert scores to ranks (lower rank = better)
def to_rank(series, higher_is_better=True):
    s = series.copy()
    if not higher_is_better:
        s = -s
    # rank 1 = best
    return s.rank(ascending=False, method="average")

# ---------- 1) RandomForest importance ----------
# Measures a feature's utility in improving the model's prediction accuracy (e.g., mean decrease in impurity).
# Captures feature interactions naturally; highly effective.
rf = RandomForestClassifier(n_estimators=500, random_state=42, class_weight="balanced")
rf.fit(Xsel, y_enc)
rf_imp = pd.Series(rf.feature_importances_, index=Xsel.columns, name="RF_Importance")
rf_rank = to_rank(rf_imp, higher_is_better=True).rename("RF_Rank")

# ---------- 2) Mutual Information ----------
# Measures statistical dependency (information gain) between a feature and the target.
# Captures non-linear relationships. Evaluates each feature independently; ignores feature interactions.
mi = mutual_info_classif(Xsel, y_enc, random_state=42, discrete_features=False)
mi_score = pd.Series(mi, index=Xsel.columns, name="MI_Score")
mi_rank = to_rank(mi_score, higher_is_better=True).rename("MI_Rank")

# ---------- 3) ANOVA F-test ----------
# (works best if roughly Gaussian/scaled; we used imputed data)
# Measures linear correlation between a feature and the target by comparing variance between groups to variance within groups.
# Assumes linear relationship and Gaussian distribution; ignores feature interactions.
F_vals, p_vals = f_classif(Xsel, y_enc)
f_score = pd.Series(F_vals, index=Xsel.columns, name="ANOVA_F")
f_rank = to_rank(f_score, higher_is_better=True).rename("ANOVA_Rank")

# ---------- Combine all rankings ----------
rank_table = pd.concat([rf_rank, mi_rank, f_rank,
                        rf_imp, mi_score, f_score], axis=1)

# OverallRank: average of available ranks (lower = better)
rank_cols = ["RF_Rank","MI_Rank","ANOVA_Rank"]
rank_table["OverallRank"] = rank_table[rank_cols].mean(axis=1)

# Sort and display top-N
N = 7
rank_table_sorted = rank_table.sort_values("OverallRank").head(N)
print("=== Top features by OverallRank (lower = better) ===")
display(rank_table_sorted)

plt.figure(figsize=(8, max(4, 0.35*N)))
rank_table_sorted.sort_values("OverallRank")["OverallRank"].plot(kind="barh")
plt.gca().invert_yaxis()
plt.title(f"Top {N} Features by Rank")
plt.xlabel("Rank (lower is better)")
plt.tight_layout()
plt.show()

topN_features = rank_table_sorted.index.tolist()
print("\nTopN feature list:", topN_features)

Features_reduced = Features[topN_features]

In [ ]:
# ============================================================================
# STEP 1: LOAD AND PREPARE DATA
# ============================================================================

def load_data(model_path, healthy_path):
    # Load model data
    df_reference = pd.read_csv(model_path)
    df_reference['Malfunction'] = df_reference['Malfunction'].astype(str)

    # Create binary label
    leakage_codes = ['C', 'D', 'E', 'F', 'G']
    df_reference['LeakageLabel'] = np.where(df_reference['Malfunction'].isin(leakage_codes), 'Combined leakage', 'Healthy')

    # Aggregate delay and efficiency columns
    delay_eff_map = {
        'Total_timing_delay': ['Brake_timing_delay_exp', 'Release_timing_delay_exp'],
        'Total_energy_delay': ['Brake_energy_delay_exp', 'Release_energy_delay_exp'],
        'Total_power_delay': ['Brake_power_delay_exp', 'Release_power_delay_exp'],
        'Total_power_efficiency': ['Brake_power_efficiency_exp', 'Release_power_efficiency_exp'],
        'Total_energy_efficiency': ['Brake_energy_effiency_exp', 'Release_energy_efficiency_exp']
    }

    for new_col, sources in delay_eff_map.items():
        df_reference[new_col] = df_reference[sources[0]] + df_reference[sources[1]]

    # Drop original columns
    cols_to_drop = [col for pair in delay_eff_map.values() for col in pair]
    df_reference.drop(columns=cols_to_drop, inplace=True)

    # Rename columns
    rename_map = {
        'Release_start_pressure_delay_exp': 'Release_start_pressure_delay',
        'Buildup_end_pressure_delay_exp': 'Buildup_end_pressure_delay',
        'Weight': 'WV_MeanPressure',
        'Brake_action': 'EmergencyBrake_action'
    }
    df_reference.rename(columns=rename_map, inplace=True)

    # Load healthy data and filter
    df_healthy = pd.read_csv(healthy_path)
    df_healthy = df_healthy[df_healthy['Non_Standard_Braking'] == 0]

    # Combine common columns
    common_cols = df_reference.columns.intersection(df_healthy.columns)
    df_combined = pd.concat([df_reference[common_cols], df_healthy[common_cols]], ignore_index=True)

    # Final label encoding
    df_combined.rename(columns={'LeakageLabel': 'label'}, inplace=True)
    df_combined['label'] = df_combined['label'].map({'Healthy': 0, 'Combined leakage': 1})

    # Convert ' sec' strings to float
    for col in df_combined.select_dtypes(include='object'):
        try:
            df_combined[col] = df_combined[col].str.replace(' sec', '', regex=False).astype(float)
        except ValueError:
            continue
    
    df = df_combined.copy()
    return df

model_path = 'model.csv'
healthy_path = 'TestBrakefinal_data.csv'

df = load_data(model_path, healthy_path)

In [ ]:
# ============================================================================
# STEP 2: DATA PREPROCESSING
# ============================================================================

def preprocess_data(df, test_size=0.2):
    """
    Preprocess data: split into train/test and separate healthy from leakage
    """
    
    mask = (df["WV_MeanPressure"] >= 2) & (df["WV_MeanPressure"] <= 3)
    df_filt = df[mask].copy()
    
    # Separate features and labels
    # X = df_filt.drop('label', axis=1)
    X = df_filt[['Total_power_efficiency']]
    y = df_filt['label']
    
    # Split data (stratified to preserve class ratio)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=42
    )
    
    # Extract only HEALTHY samples for training (for anomaly detection models)
    X_train_healthy = X_train[y_train == 0]
    
    print(f"Total samples: {len(df)}")
    print(f"Training samples: {len(X_train)} (Healthy: {sum(y_train==0)}, Leakage: {sum(y_train==1)})")
    print(f"Training samples (healthy only): {len(X_train_healthy)}")
    print(f"Test samples: {len(X_test)} (Healthy: {sum(y_test==0)}, Leakage: {sum(y_test==1)})")
    
    return X_train, X_test, y_train, y_test, X_train_healthy

[X_train, X_test, y_train, y_test, X_train_healthy] = preprocess_data(df, test_size=0.2)

X_train.head()

In [ ]:
# ============================================================================
# STEP 3: IMPUTE + FEATURE SCALING
# ============================================================================

from sklearn.impute import SimpleImputer

def scale_features(X_train, X_test, X_train_healthy):
    """
    Impute missing values by median, then standardize features.
    Returns imputed+scaled arrays, plus fitted scaler and imputer.
    """
    # 1) Median imputation (fit only on training set)
    imputer = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)
    X_train_healthy_imp = imputer.transform(X_train_healthy)

    # 2) Standardization (fit only on imputed training set)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_test_scaled = scaler.transform(X_test_imp)
    X_train_healthy_scaled = scaler.transform(X_train_healthy_imp)

    return X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer

[X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer] = scale_features(X_train, X_test, X_train_healthy)

## Model Training and Cross Validation

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score
)

# for SMOTE & its pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


# ============================================================================
# STEP 4: DEFINE ALL MODELS
# ============================================================================

def get_all_models(contamination=0.05):
    """
    Define all models to be tested
    Returns dictionary of models grouped by type
    """
    models = {
        # === ANOMALY DETECTION MODELS (train on healthy data only) ===
        'anomaly': {
            'Isolation Forest': IsolationForest(
                contamination=contamination,
                random_state=42,
                n_estimators=200,
                n_jobs=-1
            ),
            'One-Class SVM': OneClassSVM(
                nu=contamination,
                kernel='rbf',
                gamma='auto'
            ),
            'Local Outlier Factor': LocalOutlierFactor(
                contamination=contamination,
                novelty=True,
                n_neighbors=25
            )
        },
        
        # === SUPERVISED MODELS (train on imbalanced data) ===
        'supervised_imbalanced': {
            'Random Forest (Weighted)': RandomForestClassifier(
                class_weight='balanced',
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            ),
            'XGBoost (Weighted)': XGBClassifier(
                scale_pos_weight=(1100/23),  # ratio of negative/positive
                n_estimators=100,
                random_state=42,
                eval_metric='logloss'
            ),
            'Logistic Regression (Weighted)': LogisticRegression(
                class_weight='balanced',
                random_state=42,
                max_iter=1000
            ),
            # NEW: plain decision tree with class weights
            'Decision Tree (Weighted)': DecisionTreeClassifier(
                class_weight='balanced',
                random_state=42,
                max_depth=3  # you can tune this
            ),
            
        },
        
        # === SUPERVISED MODELS WITH SMOTE (train on balanced data) ===
        'supervised_smote': {
            'Random Forest (SMOTE)': RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            ),
            'XGBoost (SMOTE)': XGBClassifier(
                n_estimators=200,
                random_state=42,
                eval_metric='logloss'
            ),
            'Decision Tree (SMOTE)': DecisionTreeClassifier(
                random_state=42,
                max_depth=3
            )
        }
    }
    
    return models

# ============================================================================
# STEP 5: TRAIN ALL MODELS
# ============================================================================

def train_all_models(X_train_scaled, y_train, X_train_healthy_scaled, contamination=0.05):
    """
    Train all models and return trained models with metadata
    """
    models = get_all_models(contamination)
    trained_models = {}
    
    print("\n" + "="*60)
    print("TRAINING ALL MODELS")
    print("="*60)
    
    # Train anomaly detection models (on healthy data only)
    print("\n[1/3] Training Anomaly Detection Models...")
    for name, model in models['anomaly'].items():
        print(f"  - Training {name}...")
        model.fit(X_train_healthy_scaled)
        trained_models[name] = {
            'model': model,
            'type': 'anomaly',
            'trained': True
        }
    
    # Train supervised models on imbalanced data
    print("\n[2/3] Training Supervised Models (Imbalanced Data)...")
    for name, model in models['supervised_imbalanced'].items():
        print(f"  - Training {name}...")
        model.fit(X_train_scaled, y_train)
        trained_models[name] = {
            'model': model,
            'type': 'supervised',
            'trained': True
        }
    
    # Apply SMOTE and train supervised models
    print("\n[3/3] Training Supervised Models (with SMOTE)...")
    smote = SMOTE(random_state=42)
    X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
    print(f"  - After SMOTE: {sum(y_train_smote==0)} healthy, {sum(y_train_smote==1)} leakage")
    
    for name, model in models['supervised_smote'].items():
        print(f"  - Training {name}...")
        model.fit(X_train_smote, y_train_smote)
        trained_models[name] = {
            'model': model,
            'type': 'supervised_smote',
            'trained': True
        }
    
    print("\nAll models trained successfully!")
    return trained_models

def cross_validate_all_models_with_metrics(trained_models, X, y, k=5, out_csv=None):
    """
    Cross-validate all models with multiple metrics and return
    a ranked pandas DataFrame. Optionally saves to CSV.
    """

    # --- ensure NumPy arrays to avoid pandas indexing issues ---
    X = np.asarray(X)
    y = np.asarray(y)

    cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    records = []

    print("\n" + "="*80)
    print(f"RUNNING {k}-FOLD CROSS-VALIDATION FOR ALL MODELS (MULTI-METRIC)")
    print("="*80)

    for name, entry in trained_models.items():
        base_model = entry['model']
        mtype = entry['type']

        print(f"\n→ Evaluating {name}  (type = {mtype})")

        f1_list, prec_list, rec_list = [], [], []
        roc_list, pr_list = [], []  # ROC-AUC, PR-AUC

        for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            # --- select proper model per type ---
            if mtype == 'supervised':
                model = clone(base_model)
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)

                score_vals = None
                if hasattr(model, "predict_proba"):
                    score_vals = model.predict_proba(X_test)[:, 1]
                elif hasattr(model, "decision_function"):
                    score_vals = model.decision_function(X_test)

            elif mtype == 'supervised_smote':
                # SMOTE must be inside the CV loop to avoid leakage
                model = ImbPipeline([
                    ('smote', SMOTE(random_state=42)),
                    ('clf', clone(base_model))
                ])
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)

                score_vals = None
                clf = model.named_steps['clf']
                if hasattr(clf, "predict_proba"):
                    score_vals = clf.predict_proba(X_test)[:, 1]
                elif hasattr(clf, "decision_function"):
                    score_vals = clf.decision_function(X_test)


            elif mtype == 'anomaly':
                # Train only on healthy samples in the training fold
                X_train_healthy = X_train[y_train == 0]
                model = clone(base_model)
                model.fit(X_train_healthy)

                raw_pred = model.predict(X_test)          # -1 = anomaly, 1 = normal
                y_pred = (raw_pred == -1).astype(int)     # 1 = leakage
                score_vals = None  # no continuous scores here

            else:
                raise ValueError(f"Unknown model type: {mtype}")

            # --- basic classification metrics ---
            f1_list.append(f1_score(y_test, y_pred, zero_division=0))
            prec_list.append(precision_score(y_test, y_pred, zero_division=0))
            rec_list.append(recall_score(y_test, y_pred, zero_division=0))

            # --- ROC-AUC and PR-AUC (only if we have scores and both classes present) ---
            if score_vals is not None and len(np.unique(y_test)) == 2:
                try:
                    roc_list.append(roc_auc_score(y_test, score_vals))
                    pr_list.append(average_precision_score(y_test, score_vals))
                except ValueError:
                    # can still fail if only 1 class in y_test after all
                    pass

        # --- aggregate results for this model ---
        record = {
            'Model': name,
            'Type': mtype,
            'F1_mean':   np.mean(f1_list),
            'F1_std':    np.std(f1_list),
            'Precision_mean': np.mean(prec_list),
            'Recall_mean':    np.mean(rec_list),
            'ROC_AUC_mean':  np.mean(roc_list) if len(roc_list) > 0 else np.nan,
            'PR_AUC_mean':   np.mean(pr_list)  if len(pr_list) > 0 else np.nan,
        }
        records.append(record)

        print(f"  F1      per-fold: {np.round(f1_list, 3)}  | mean = {record['F1_mean']:.3f}")
        print(f"  Prec    per-fold: {np.round(prec_list, 3)} | mean = {record['Precision_mean']:.3f}")
        print(f"  Recall  per-fold: {np.round(rec_list, 3)} | mean = {record['Recall_mean']:.3f}")
        if not np.isnan(record['ROC_AUC_mean']):
            print(f"  ROC-AUC mean   = {record['ROC_AUC_mean']:.3f}")
        if not np.isnan(record['PR_AUC_mean']):
            print(f"  PR-AUC  mean   = {record['PR_AUC_mean']:.3f}")

    # --- build table and rank by mean F1 ---
    df_results = pd.DataFrame(records)
    df_results_sorted = df_results.sort_values(by='F1_mean', ascending=False).reset_index(drop=True)

    print("\n" + "="*80)
    print("CROSS-VALIDATION SUMMARY (RANKED BY MEAN F1)")
    print("="*80)
    print(df_results_sorted)

    if out_csv is not None:
        df_results_sorted.to_csv(out_csv, index=False)
        print(f"\nSaved CV summary to: {out_csv}")

    return df_results_sorted



trained_models = train_all_models(X_train_scaled, y_train, X_train_healthy_scaled, contamination=0.05)

cv_summary = cross_validate_all_models_with_metrics(
    trained_models,
    X_train_scaled,
    y_train,
    k=5,
    out_csv="cv_results_summary.csv"
)

### Figure plotting of Cross Validation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Optional: to make plots look nicer
plt.style.use('seaborn-v0_8')  # or comment out if you prefer default

# --- Prepare data ---
models = cv_summary['Model'].values
f1_mean = cv_summary['F1_mean'].values
f1_std  = cv_summary['F1_std'].values
types   = cv_summary['Type'].values  # 'supervised', 'supervised_smote', 'anomaly'

# Assign a color per type
type_to_color = {
    'supervised': '#1f77b4',        # blue
    'supervised_smote': '#2ca02c',  # green
    'anomaly': '#d62728'            # red
}
colors = [type_to_color[t] for t in types]

# --- Create bar chart ---
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(models))

bars = ax.bar(x, f1_mean, yerr=f1_std, capsize=5, color=colors, alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45, ha='right')
ax.set_ylabel('F1-score (mean ± std)')
ax.set_title('Cross-validated F1-score per Model')

# Build custom legend
handles = []
labels  = []
for t, c in type_to_color.items():
    handles.append(plt.Rectangle((0, 0), 1, 1, color=c))
    labels.append(t)
ax.legend(handles, labels, title='Model Type')

ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import clone

# Choose the model name you want to visualize
best_model_name = "Decision Tree (SMOTE)"  # or "Random Forest (Weighted)", etc.

base_model = trained_models[best_model_name]['model']
mtype      = trained_models[best_model_name]['type']

X = np.asarray(X_train_scaled)
y = np.asarray(y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- Build estimator depending on type ---
if mtype == 'supervised_smote':
    estimator = ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('clf', clone(base_model))
    ])
elif mtype == 'supervised':
    estimator = clone(base_model)
else:
    raise ValueError("Confusion matrix is more meaningful for supervised models.")

# --- Cross-validated predictions (each sample predicted from a different fold) ---
y_pred_cv = cross_val_predict(estimator, X, y, cv=cv)

# --- Confusion matrix ---
cm = confusion_matrix(y, y_pred_cv)  # [[TN, FP], [FN, TP]]

# Optionally normalize to percentages
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))

# Raw counts
im0 = ax[0].imshow(cm, cmap='coolwarm')
ax[0].set_title('Confusion Matrix (Counts)')
ax[0].set_xticks([0, 1]); ax[0].set_yticks([0, 1])
ax[0].set_xticklabels(['Pred Healthy', 'Pred Leakage'])
ax[0].set_yticklabels(['True Healthy', 'True Leakage'])

for i in range(2):
    for j in range(2):
        text_color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax[0].text(j, i, cm[i, j], ha='center', va='center', color=text_color)
plt.colorbar(im0, ax=ax[0], fraction=0.046, pad=0.04)

# Normalized
im1 = ax[1].imshow(cm_norm, cmap='coolwarm', vmin=0, vmax=1)
ax[1].set_title('Confusion Matrix (Row-normalized)')
ax[1].set_xticks([0, 1]); ax[1].set_yticks([0, 1])
ax[1].set_xticklabels(['Pred Healthy', 'Pred Leakage'])
ax[1].set_yticklabels(['True Healthy', 'True Leakage'])
for i in range(2):
    for j in range(2):
        text_color = 'white' if cm_norm[i, j] > 0.5 else 'black'
        ax[1].text(j, i, f"{cm_norm[i, j]:.2f}", ha='center', va='center', color=text_color)

plt.colorbar(im1, ax=ax[1], fraction=0.046, pad=0.04)

plt.suptitle(f'Confusion Matrix for {best_model_name}', y=1.02)
plt.tight_layout()
for axis in ax:
    axis.grid(False)  # disables gridlines
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.base import clone

# Pick one model to inspect
pr_model_name = "Random Forest (Weighted)"   # or "Decision Tree (SMOTE)", etc.

base_model = trained_models[pr_model_name]['model']
mtype      = trained_models[pr_model_name]['type']

X = np.asarray(X_train_scaled)
y = np.asarray(y_train)

# --- Build estimator depending on type ---
if mtype == 'supervised':
    estimator = clone(base_model)

elif mtype == 'supervised_smote':
    estimator = ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('clf', clone(base_model))
    ])

else:
    raise ValueError("Use a supervised model for cross-validated PR curve.")

# --- Cross-validated out-of-fold scores ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# This gives, for each sample, a prediction made by a model
# that did NOT see that sample during training
y_scores = cross_val_predict(
    estimator, X, y,
    cv=cv,
    method="predict_proba"
)[:, 1]

# --- PR curve on pooled CV scores ---
precision, recall, thresholds = precision_recall_curve(y, y_scores)
avg_prec = average_precision_score(y, y_scores)

plt.figure(figsize=(6, 5))
plt.plot(recall, precision, label=f'PR curve (AP = {avg_prec:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title(f'Cross-validated Precision–Recall Curve: {pr_model_name}')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()


### Precision Recall curves (Sensitive to imbalance data)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.base import clone

# -------------------------------------------------------
# 1. Choose models to plot
# -------------------------------------------------------
models_to_plot = [
    "Random Forest (Weighted)",
    "Decision Tree (SMOTE)",
    "Logistic Regression (Weighted)",
    "XGBoost (SMOTE)",
]

X = np.asarray(X_train_scaled)
y = np.asarray(y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

plt.figure(figsize=(7, 6))

for name in models_to_plot:
    entry = trained_models[name]
    base_model = entry['model']
    mtype      = entry['type']

    # --- build estimator depending on type ---
    if mtype == 'supervised':
        estimator = clone(base_model)

    elif mtype == 'supervised_smote':
        estimator = ImbPipeline([
            ('smote', SMOTE(random_state=42)),
            ('clf', clone(base_model))
        ])

    else:
        print(f"Skipping {name} (type '{mtype}' not supported for PR curve).")
        continue

    # --- cross-validated scores (out-of-fold) ---
    y_scores = cross_val_predict(
        estimator, X, y,
        cv=cv,
        method="predict_proba"
    )[:, 1]

    precision, recall, _ = precision_recall_curve(y, y_scores)
    ap = average_precision_score(y, y_scores)

    plt.plot(recall, precision, label=f"{name} (AP = {ap:.3f})")

# baseline: proportion of positives (random classifier)
pos_ratio = y.mean()
plt.hlines(pos_ratio, 0, 1, colors='gray', linestyles='--',
           label=f"Baseline (pos ratio = {pos_ratio:.2f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Cross-validated Precision–Recall Curves")
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()


### ROC Curves

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.model_selection import cross_val_predict

plt.figure(figsize=(7, 6))

models_to_plot = [
    "Random Forest (Weighted)",
    "Decision Tree (SMOTE)",
    "Logistic Regression (Weighted)",
    "XGBoost (SMOTE)",
]

X = np.asarray(X_train_scaled)
y = np.asarray(y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name in models_to_plot:
    entry = trained_models[name]
    base_model = entry['model']
    mtype      = entry['type']

    # Build estimator
    if mtype == 'supervised':
        estimator = clone(base_model)
    elif mtype == 'supervised_smote':
        estimator = ImbPipeline([
            ('smote', SMOTE(random_state=42)),
            ('clf', clone(base_model))
        ])
    else:
        print(f"Skipping anomaly model: {name}")
        continue

    # CV predicted probabilities
    y_scores = cross_val_predict(
        estimator, X, y,
        cv=cv,
        method="predict_proba"
    )[:, 1]

    fpr, tpr, _ = roc_curve(y, y_scores)
    auc = roc_auc_score(y, y_scores)

    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})", lw=2)

# Random baseline
plt.plot([0, 1], [0, 1], 'k--', label="Random classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Cross-validated ROC Curves for Multiple Models")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd

# Convert to arrays
X = np.asarray(X_train).ravel()
y = np.asarray(y_train)

# Build a simple DataFrame
df_plot = pd.DataFrame({
    'TPE': X,
    'Label': y
})

df_plot['Label'] = df_plot['Label'].map({0: 'Healthy', 1: 'Leakage'})


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ============= LEFT: KDE PLOT =============
sns.kdeplot(
    data=df_plot[df_plot['Label'] == 'Healthy'],
    x='TPE', ax=axes[0],
    fill=True, alpha=0.4,
    linewidth=2, label='Healthy', color='#1f77b4'
)
sns.kdeplot(
    data=df_plot[df_plot['Label'] == 'Leakage'],
    x='TPE', ax=axes[0],
    fill=True, alpha=0.4,
    linewidth=2, label='Leakage', color='#d62728'
)

axes[0].set_title("KDE Distribution", fontsize=14)
axes[0].set_xlabel("Total Power Efficiency (scaled)")
axes[0].set_ylabel("Density")
axes[0].grid(alpha=0.3)
axes[0].legend()

# ============= RIGHT: BOXPLOT =============
sns.boxplot(
    data=df_plot,
    x='Label', y='TPE',
    ax=axes[1],
    palette=['#1f77b4', '#d62728'],
    width=0.5
)

axes[1].set_title("Boxplot by Class", fontsize=14)
axes[1].set_xlabel("")
axes[1].set_ylabel("Total Power Efficiency (scaled)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


### Threshold Value for the top 2 models

In [ ]:
from sklearn.tree import export_text

tree = trained_models['Decision Tree (SMOTE)']['model']
print(export_text(tree, feature_names=['TPE']))

original_threshold = scaler.inverse_transform([[-0.42]])
print(original_threshold)



In [ ]:
import numpy as np

rf = trained_models['Random Forest (Weighted)']['model']
probs = rf.predict_proba(X_train_scaled)[:, 1]

# ROC curve gives candidate thresholds
from sklearn.metrics import precision_recall_curve
precision, recall, thresholds = precision_recall_curve(y_train, probs)

# Find threshold that maximizes F1 on training set
f1_scores = 2 * precision * recall / (precision + recall)
best_idx = np.nanargmax(f1_scores)
best_threshold = thresholds[best_idx]
# 1. Extract all unique TPE values from training data
tpe_vals = np.sort(X_train_scaled.ravel())

# 2. Create a very fine grid for smoother threshold detection
grid = np.linspace(tpe_vals.min(), tpe_vals.max(), 500)

# 3. Predict probabilities for all grid points
probs = rf.predict_proba(grid.reshape(-1, 1))[:, 1]

# 4. Find where probability crosses the chosen threshold
idx = np.argmin(np.abs(probs - best_threshold))
tpe_threshold_scaled = grid[idx]
tpe_threshold_original = scaler.inverse_transform([[tpe_threshold_scaled]])

print("Best F1 threshold =", best_threshold)
print("Scaled TPE threshold =", tpe_threshold_scaled)
print("Original TPE threshold =", tpe_threshold_original[0][0])

### Comparison Table of F1 Score, Precision, Recall, and ROC value

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
from sklearn.base import clone
import pandas as pd
import numpy as np

def crossvalidated_metrics_table(trained_models, X, y):
    X = np.asarray(X)
    y = np.asarray(y)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    rows = []

    for name, entry in trained_models.items():
        base_model = entry['model']
        mtype      = entry['type']

        # ---- Build estimator properly ----
        if mtype == "supervised":
            estimator = clone(base_model)

        elif mtype == "supervised_smote":
            estimator = ImbPipeline([
                ('smote', SMOTE(random_state=42)),
                ('clf', clone(base_model))
            ])

        elif mtype == "anomaly":
            # anomaly models require training only on healthy subset inside each fold
            # so they cannot use cross_val_predict directly
            # we will do manual CV predictions
            y_pred = np.zeros_like(y)
            
            for train_idx, test_idx in cv.split(X, y):
                X_train, X_test = X[train_idx], X[test_idx]
                y_train = y[train_idx]

                X_train_healthy = X_train[y_train == 0]

                model = clone(base_model)
                model.fit(X_train_healthy)

                out = model.predict(X_test)        # -1=anomaly, 1=normal
                y_pred[test_idx] = (out == -1).astype(int)  # leakage=1
        else:
            continue

        # ---- Supervised models → get out-of-fold predictions ----
        if mtype != "anomaly":
            y_pred_proba = cross_val_predict(
                estimator, X, y, cv=cv, method='predict_proba'
            )[:, 1]
            y_pred = (y_pred_proba >= 0.5).astype(int)

        # ---- confusion matrix ----
        tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

        # ---- metrics ----
        precision = precision_score(y, y_pred, zero_division=0)
        recall    = recall_score(y, y_pred, zero_division=0)
        f1        = f1_score(y, y_pred, zero_division=0)

        # ROC-AUC for supervised
        if mtype != "anomaly":
            rocauc = roc_auc_score(y, y_pred_proba)
        else:
            rocauc = np.nan

        rows.append({
            "Model": name,
            "Type": mtype,
            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1,
            "ROC-AUC": rocauc,
            "True Positives": tp,
            "False Positives": fp,
            "False Negatives": fn,
            "True Negatives": tn,
        })

    df = pd.DataFrame(rows).sort_values(by="F1-Score", ascending=False)
    return df

cv_results_full = crossvalidated_metrics_table(
    trained_models, X_train_scaled, y_train
)

cv_results_full


# (Not used) Just example of pipeline could be

In [ ]:
# ============================================================================
# STEP 6: EVALUATE ALL MODELS
# ============================================================================

def evaluate_all_models(trained_models, X_test_scaled, y_test):
    """
    Evaluate all models and collect results
    """
    results = []
    all_predictions = {}
    all_scores = {}
    
    print("\n" + "="*60)
    print("EVALUATING ALL MODELS")
    print("="*60)
    
    for name, model_info in trained_models.items():
        model = model_info['model']
        model_type = model_info['type']
        
        print(f"\nEvaluating: {name}")
        
        # Make predictions
        if model_type == 'anomaly':
            y_pred = model.predict(X_test_scaled)
            # Convert: -1 (anomaly) -> 1 (leakage), 1 (normal) -> 0 (healthy)
            y_pred = np.where(y_pred == -1, 1, 0)
            
            # Get anomaly scores
            scores = model.score_samples(X_test_scaled)
            scores = -scores  # Invert so higher = more anomalous
        else:
            y_pred = model.predict(X_test_scaled)
            # Get probability scores for positive class
            if hasattr(model, 'predict_proba'):
                scores = model.predict_proba(X_test_scaled)[:, 1]
            else:
                scores = model.decision_function(X_test_scaled)
        
        all_predictions[name] = y_pred
        all_scores[name] = scores
        
        # Calculate metrics
        cm = confusion_matrix(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        
        try:
            roc_auc = roc_auc_score(y_test, scores)
        except:
            roc_auc = 0.0
        
        # Store results
        results.append({
            'Model': name,
            'Type': model_type,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1,
            'ROC-AUC': roc_auc,
            'True Positives': cm[1,1] if cm.shape[0] > 1 else 0,
            'False Positives': cm[0,1] if cm.shape[0] > 1 else 0,
            'False Negatives': cm[1,0] if cm.shape[0] > 1 else 0,
            'True Negatives': cm[0,0]
        })
        
        print(f"  Precision: {precision:.3f}, Recall: {recall:.3f}, F1: {f1:.3f}, ROC-AUC: {roc_auc:.3f}")
    
    # Create results DataFrame
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('F1-Score', ascending=False).reset_index(drop=True)
    
    return results_df, all_predictions, all_scores


# ============================================================================
# STEP 7: VISUALIZE COMPARISON
# ============================================================================

def visualize_comparison(results_df, all_predictions, all_scores, y_test):
    """
    Create comprehensive comparison visualizations
    """
    n_models = len(results_df)
    
    # Create figure with multiple subplots
    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    # 1. Model Performance Comparison (Bar Chart)
    ax1 = fig.add_subplot(gs[0, :])
    x = np.arange(n_models)
    width = 0.2
    
    ax1.bar(x - 1.5*width, results_df['Precision'], width, label='Precision', alpha=0.8)
    ax1.bar(x - 0.5*width, results_df['Recall'], width, label='Recall', alpha=0.8)
    ax1.bar(x + 0.5*width, results_df['F1-Score'], width, label='F1-Score', alpha=0.8)
    ax1.bar(x + 1.5*width, results_df['ROC-AUC'], width, label='ROC-AUC', alpha=0.8)
    
    ax1.set_xlabel('Models', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax1.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(results_df['Model'], rotation=45, ha='right')
    ax1.legend(loc='upper left')
    ax1.grid(axis='y', alpha=0.3)
    ax1.set_ylim([0, 1.1])
    
    # 2. F1-Score Ranking
    ax2 = fig.add_subplot(gs[1, 0])
    colors = plt.cm.RdYlGn(results_df['F1-Score'])
    ax2.barh(results_df['Model'], results_df['F1-Score'], color=colors)
    ax2.set_xlabel('F1-Score', fontweight='bold')
    ax2.set_title('F1-Score Ranking', fontsize=12, fontweight='bold')
    ax2.grid(axis='x', alpha=0.3)
    
    # 3. Recall vs Precision Scatter
    ax3 = fig.add_subplot(gs[1, 1])
    scatter = ax3.scatter(results_df['Recall'], results_df['Precision'], 
                         s=results_df['F1-Score']*500, alpha=0.6, 
                         c=results_df['ROC-AUC'], cmap='viridis')
    for idx, row in results_df.iterrows():
        ax3.annotate(row['Model'], (row['Recall'], row['Precision']), 
                    fontsize=8, alpha=0.7, ha='center')
    ax3.set_xlabel('Recall', fontweight='bold')
    ax3.set_ylabel('Precision', fontweight='bold')
    ax3.set_title('Precision vs Recall (size=F1, color=ROC-AUC)', fontsize=12, fontweight='bold')
    ax3.grid(alpha=0.3)
    plt.colorbar(scatter, ax=ax3, label='ROC-AUC')
    
    # 4. Confusion Matrix Heatmap for Top 3 Models
    top_3_models = results_df.head(3)['Model'].tolist()
    
    for idx, model_name in enumerate(top_3_models):
        ax = fig.add_subplot(gs[1, 2]) if idx == 0 else fig.add_subplot(gs[2, idx-1])
        
        y_pred = all_predictions[model_name]
        cm = confusion_matrix(y_test, y_pred)
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                   xticklabels=['Healthy', 'Leakage'],
                   yticklabels=['Healthy', 'Leakage'],
                   cbar=False)
        ax.set_title(f'{model_name}\nConfusion Matrix', fontsize=10, fontweight='bold')
        ax.set_ylabel('True Label')
        ax.set_xlabel('Predicted Label')
    
    # 5. ROC Curves Comparison
    ax5 = fig.add_subplot(gs[2, 2])
    
    for model_name in results_df.head(5)['Model']:  # Top 5 models
        scores = all_scores[model_name]
        try:
            fpr, tpr, _ = roc_curve(y_test, scores)
            auc = roc_auc_score(y_test, scores)
            ax5.plot(fpr, tpr, label=f'{model_name} (AUC={auc:.3f})', linewidth=2)
        except:
            pass
    
    ax5.plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1)
    ax5.set_xlabel('False Positive Rate', fontweight='bold')
    ax5.set_ylabel('True Positive Rate', fontweight='bold')
    ax5.set_title('ROC Curves (Top 5 Models)', fontsize=12, fontweight='bold')
    ax5.legend(loc='lower right', fontsize=8)
    ax5.grid(alpha=0.3)
    
    plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
    print("\n✓ Comparison visualization saved as 'model_comparison.png'")
    plt.show()

# ============================================================================
# SEPARATE BENCHMARKS
# ============================================================================

def visualize_supervised_benchmark(results_df, all_predictions, all_scores, y_test, title_suffix=""):
    """
    Visualize ONLY supervised models (weighted + SMOTE).
    """
    sup_df = results_df[results_df['Type'] != 'anomaly'].copy()
    if sup_df.empty:
        print("No supervised models to visualize.")
        return

    n_models = len(sup_df)
    fig = plt.figure(figsize=(18, 10))
    gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)

    # 1) Performance bars
    ax1 = fig.add_subplot(gs[0, :])
    x = np.arange(n_models)
    width = 0.22

    ax1.bar(x - 1.5*width, sup_df['Precision'], width, label='Precision', alpha=0.85)
    ax1.bar(x - 0.5*width, sup_df['Recall'], width, label='Recall', alpha=0.85)
    ax1.bar(x + 0.5*width, sup_df['F1-Score'], width, label='F1-Score', alpha=0.85)
    ax1.bar(x + 1.5*width, sup_df['ROC-AUC'], width, label='ROC-AUC', alpha=0.85)
    ax1.set_xticks(x)
    ax1.set_xticklabels(sup_df['Model'], rotation=40, ha='right')
    ax1.set_ylim([0, 1.1])
    ax1.set_title(f"Supervised Models — Performance{title_suffix}")
    ax1.grid(axis='y', alpha=0.3)
    ax1.legend(loc='upper left')

    # 2) Confusion matrix of the best 2 supervised models by F1
    top_models = sup_df.sort_values('F1-Score', ascending=False).head(2)['Model'].tolist()
    for i, m in enumerate(top_models):
        ax = fig.add_subplot(gs[1, i])
        y_pred = all_predictions[m]
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                    xticklabels=['Healthy', 'Leakage'], yticklabels=['Healthy', 'Leakage'], ax=ax)
        ax.set_title(f'{m} — Confusion Matrix')
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')

    # 3) ROC curves (top 5 by F1)
    # put on ax in first row right? we've used both bottom subplots; ROC can go in ax1? We'll add overlay on ax1 is crowded:
    # Instead save fig and break out.
    plt.savefig('supervised_benchmark.png', dpi=300, bbox_inches='tight')
    print("✓ Supervised benchmark saved as 'supervised_benchmark.png'")
    plt.show()

    # Separate ROC figure
    fig2, ax = plt.subplots(figsize=(8, 6))
    for m in sup_df.sort_values('F1-Score', ascending=False).head(5)['Model']:
        scores = all_scores[m]
        try:
            fpr, tpr, _ = roc_curve(y_test, scores)
            auc = roc_auc_score(y_test, scores)
            ax.plot(fpr, tpr, label=f'{m} (AUC={auc:.3f})', linewidth=2)
        except Exception:
            pass
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    ax.set_title(f"Supervised Models — ROC Curves{title_suffix}")
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(alpha=0.3)
    plt.savefig('supervised_roc.png', dpi=300, bbox_inches='tight')
    print("✓ Supervised ROC curves saved as 'supervised_roc.png'")
    plt.show()


def visualize_anomaly_benchmark(results_df, all_predictions, all_scores, y_test, title_suffix=""):
    """
    Visualize ONLY anomaly detection models (IF, OCSVM, LOF).
    Includes: metric bars, score histograms by class, ROC curves.
    """
    ano_df = results_df[results_df['Type'] == 'anomaly'].copy()
    if ano_df.empty:
        print("No anomaly models to visualize.")
        return

    # --- 1) Bars of Precision / Recall / F1 (they might be zero if no positives predicted)
    fig = plt.figure(figsize=(14, 6))
    ax1 = fig.add_subplot(1, 1, 1)
    x = np.arange(len(ano_df)); width = 0.25
    ax1.bar(x - width, ano_df['Precision'], width, label='Precision', alpha=0.9)
    ax1.bar(x,         ano_df['Recall'],    width, label='Recall',    alpha=0.9)
    ax1.bar(x + width, ano_df['F1-Score'],  width, label='F1-Score',  alpha=0.9)
    ax1.set_xticks(x)
    ax1.set_xticklabels(ano_df['Model'], rotation=20, ha='right')
    ax1.set_ylim([0, 1.05])
    ax1.set_title(f"Anomaly Models — Discrete Metrics (current thresholds){title_suffix}")
    ax1.grid(axis='y', alpha=0.3)
    ax1.legend()
    plt.savefig('anomaly_metrics.png', dpi=300, bbox_inches='tight')
    print("✓ Anomaly metrics saved as 'anomaly_metrics.png'")
    plt.show()

    # --- 2) Score histograms by class for each anomaly model
    # Scores are higher = more anomalous (you inverted IsolationForest already)
    for m in ano_df['Model']:
        scores = all_scores[m]
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.hist(scores[y_test == 0], bins=30, alpha=0.6, label='Healthy', density=True)
        ax.hist(scores[y_test == 1], bins=30, alpha=0.6, label='Leakage', density=True)
        ax.set_title(f'{m} — Anomaly Score Distributions{title_suffix}')
        ax.set_xlabel('Anomaly score'); ax.set_ylabel('Density')
        ax.legend()
        ax.grid(alpha=0.3)
        fname = f"{m.replace(' ', '_').lower()}_score_hist.png"
        plt.savefig(fname, dpi=300, bbox_inches='tight')
        print(f"✓ Score histogram saved as '{fname}'")
        plt.show()

    # --- 3) ROC curves for anomaly models (using scores)
    fig, ax = plt.subplots(figsize=(8, 6))
    for m in ano_df['Model']:
        scores = all_scores[m]
        try:
            fpr, tpr, _ = roc_curve(y_test, scores)
            auc = roc_auc_score(y_test, scores)
            ax.plot(fpr, tpr, label=f'{m} (AUC={auc:.3f})', linewidth=2)
        except Exception:
            pass
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    ax.set_title(f"Anomaly Models — ROC Curves{title_suffix}")
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(alpha=0.3)
    plt.savefig('anomaly_roc.png', dpi=300, bbox_inches='tight')
    print("✓ Anomaly ROC curves saved as 'anomaly_roc.png'")
    plt.show()

# ============================================================================
# STEP 8: GENERATE DETAILED REPORT
# ============================================================================

def generate_report(results_df):
    """
    Generate detailed text report
    """
    print("\n" + "="*80)
    print("COMPREHENSIVE MODEL COMPARISON REPORT")
    print("="*80)
    
    print("\n📊 COMPLETE RESULTS TABLE:")
    print("-"*80)
    print(results_df.to_string(index=False))
    
    print("\n" + "="*80)
    print("🏆 TOP 3 MODELS")
    print("="*80)
    
    for idx, row in results_df.head(3).iterrows():
        print(f"\n#{idx+1}: {row['Model']}")
        print(f"  • Type: {row['Type']}")
        print(f"  • Precision: {row['Precision']:.3f} (of predicted leakages, {row['Precision']*100:.1f}% were correct)")
        print(f"  • Recall: {row['Recall']:.3f} (detected {row['Recall']*100:.1f}% of actual leakages)")
        print(f"  • F1-Score: {row['F1-Score']:.3f}")
        print(f"  • ROC-AUC: {row['ROC-AUC']:.3f}")
        print(f"  • True Positives: {row['True Positives']} | False Positives: {row['False Positives']}")
        print(f"  • False Negatives: {row['False Negatives']} | True Negatives: {row['True Negatives']}")
    
    print("\n" + "="*80)
    print("💡 RECOMMENDATIONS")
    print("="*80)
    
    best_model = results_df.iloc[0]
    
    if best_model['Recall'] > 0.8:
        print(f"✓ {best_model['Model']} shows excellent recall ({best_model['Recall']:.1%})")
        print("  → Good at catching leakages, recommended for safety-critical applications")
    elif best_model['Precision'] > 0.8:
        print(f"✓ {best_model['Model']} shows excellent precision ({best_model['Precision']:.1%})")
        print("  → Low false alarms, recommended for cost-sensitive applications")
    
    if best_model['F1-Score'] > 0.7:
        print(f"✓ Strong overall performance (F1={best_model['F1-Score']:.3f})")
    else:
        print("⚠ Consider collecting more leakage samples to improve model performance")
    
    print("\n" + "="*80)

def generate_grouped_report(results_df):
    print("\n" + "="*80)
    print("COMPREHENSIVE MODEL COMPARISON REPORT (Grouped)")
    print("="*80)

    if not results_df[results_df['Type'] != 'anomaly'].empty:
        print("\n--- SUPERVISED MODELS ---")
        print(results_df[results_df['Type'] != 'anomaly'].sort_values('F1-Score', ascending=False).to_string(index=False))

    if not results_df[results_df['Type'] == 'anomaly'].empty:
        print("\n--- ANOMALY MODELS ---")
        print(results_df[results_df['Type'] == 'anomaly'].sort_values('F1-Score', ascending=False).to_string(index=False))

# ============================================================================
# MAIN PIPELINE
# ============================================================================

def main():
    """
    Complete pipeline execution with multiple models
    """
    print("="*80)
    print("MULTI-MODEL ANOMALY DETECTION PIPELINE")
    print("="*80)
    
    # Step 1: Load data
    print("\n[1/7] Loading data...")
    df = load_data("model.csv", "TestBrakefinal_data_healthy.csv")  # Replace with your data loading
    
    # Step 2: Preprocess
    print("\n[2/7] Preprocessing data...")
    X_train, X_test, y_train, y_test, X_train_healthy = preprocess_data(df)
    
    # Step 3: Impute + Scale features
    print("\n[3/7] Imputing (median) and scaling features...")
    X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer = scale_features(
    X_train, X_test, X_train_healthy
    )
    
    # Step 4: Train all models
    print("\n[4/7] Training all models...")
    contamination_rate = 23 / 1123
    trained_models = train_all_models(
        X_train_scaled, y_train, X_train_healthy_scaled, contamination_rate
    )
    
    # Step 5: Evaluate all models
    print("\n[5/7] Evaluating all models...")
    results_df, all_predictions, all_scores = evaluate_all_models(
        trained_models, X_test_scaled, y_test
    )
    
    # Step 6: Visualize comparison (separate)
    print("\n[6/7] Creating separate benchmarks...")
    visualize_supervised_benchmark(results_df, all_predictions, all_scores, y_test, title_suffix="")
    visualize_anomaly_benchmark(results_df, all_predictions, all_scores, y_test, title_suffix="")

    # Step 7: Generate report
    print("\n[7/7] Generating detailed report...")
    generate_report(results_df)
    

    # Step 8: Generate grouped report
    print("\n[7/7] Generating grouped report...")
    generate_grouped_report(results_df)
        
    print("\n" + "="*80)
    print("✓ PIPELINE COMPLETE!")
    print("="*80)
    
    return trained_models, results_df, scaler

# ============================================================================
# RUN PIPELINE
# ============================================================================

if __name__ == "__main__":
    trained_models, results_df, scaler = main()
    
    # Access best model
    best_model_name = results_df.iloc[0]['Model']
    best_model = trained_models[best_model_name]['model']
    
    print(f"\n🎯 Best Model: {best_model_name}")
    print(f"   F1-Score: {results_df.iloc[0]['F1-Score']:.3f}")
    
    # Save best model (optional)
    # import joblib
    # joblib.dump(best_model, f'{best_model_name.replace(" ", "_")}_model.pkl')
    # joblib.dump(scaler, 'scaler.pkl')
    # joblib.dump(imputer, 'imputer.pkl')

# (Not used) Just example of pipeline could be

In [ ]:
# ============================================================================
# STEP 7: VISUALIZE COMPARISON
# ============================================================================

def visualize_comparison(results_df, all_predictions, all_scores, y_test):
    """
    Create comprehensive comparison visualizations
    """
    n_models = len(results_df)
    
    # Create figure with multiple subplots
    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    # 1. Model Performance Comparison (Bar Chart)
    ax1 = fig.add_subplot(gs[0, :])
    x = np.arange(n_models)
    width = 0.2
    
    ax1.bar(x - 1.5*width, results_df['Precision'], width, label='Precision', alpha=0.8)
    ax1.bar(x - 0.5*width, results_df['Recall'], width, label='Recall', alpha=0.8)
    ax1.bar(x + 0.5*width, results_df['F1-Score'], width, label='F1-Score', alpha=0.8)
    ax1.bar(x + 1.5*width, results_df['ROC-AUC'], width, label='ROC-AUC', alpha=0.8)
    
    ax1.set_xlabel('Models', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax1.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(results_df['Model'], rotation=45, ha='right')
    ax1.legend(loc='upper left')
    ax1.grid(axis='y', alpha=0.3)
    ax1.set_ylim([0, 1.1])
    
    # 2. F1-Score Ranking
    ax2 = fig.add_subplot(gs[1, 0])
    colors = plt.cm.RdYlGn(results_df['F1-Score'])
    ax2.barh(results_df['Model'], results_df['F1-Score'], color=colors)
    ax2.set_xlabel('F1-Score', fontweight='bold')
    ax2.set_title('F1-Score Ranking', fontsize=12, fontweight='bold')
    ax2.grid(axis='x', alpha=0.3)
    
    # 3. Recall vs Precision Scatter
    ax3 = fig.add_subplot(gs[1, 1])
    scatter = ax3.scatter(results_df['Recall'], results_df['Precision'], 
                         s=results_df['F1-Score']*500, alpha=0.6, 
                         c=results_df['ROC-AUC'], cmap='viridis')
    for idx, row in results_df.iterrows():
        ax3.annotate(row['Model'], (row['Recall'], row['Precision']), 
                    fontsize=8, alpha=0.7, ha='center')
    ax3.set_xlabel('Recall', fontweight='bold')
    ax3.set_ylabel('Precision', fontweight='bold')
    ax3.set_title('Precision vs Recall (size=F1, color=ROC-AUC)', fontsize=12, fontweight='bold')
    ax3.grid(alpha=0.3)
    plt.colorbar(scatter, ax=ax3, label='ROC-AUC')
    
    # 4. Confusion Matrix Heatmap for Top 3 Models
    top_3_models = results_df.head(3)['Model'].tolist()
    
    for idx, model_name in enumerate(top_3_models):
        ax = fig.add_subplot(gs[1, 2]) if idx == 0 else fig.add_subplot(gs[2, idx-1])
        
        y_pred = all_predictions[model_name]
        cm = confusion_matrix(y_test, y_pred)
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                   xticklabels=['Healthy', 'Leakage'],
                   yticklabels=['Healthy', 'Leakage'],
                   cbar=False)
        ax.set_title(f'{model_name}\nConfusion Matrix', fontsize=10, fontweight='bold')
        ax.set_ylabel('True Label')
        ax.set_xlabel('Predicted Label')
    
    # 5. ROC Curves Comparison
    ax5 = fig.add_subplot(gs[2, 2])
    
    for model_name in results_df.head(5)['Model']:  # Top 5 models
        scores = all_scores[model_name]
        try:
            fpr, tpr, _ = roc_curve(y_test, scores)
            auc = roc_auc_score(y_test, scores)
            ax5.plot(fpr, tpr, label=f'{model_name} (AUC={auc:.3f})', linewidth=2)
        except:
            pass
    
    ax5.plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1)
    ax5.set_xlabel('False Positive Rate', fontweight='bold')
    ax5.set_ylabel('True Positive Rate', fontweight='bold')
    ax5.set_title('ROC Curves (Top 5 Models)', fontsize=12, fontweight='bold')
    ax5.legend(loc='lower right', fontsize=8)
    ax5.grid(alpha=0.3)
    
    plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
    print("\n✓ Comparison visualization saved as 'model_comparison.png'")
    plt.show()

# ============================================================================
# SEPARATE BENCHMARKS
# ============================================================================

def visualize_supervised_benchmark(results_df, all_predictions, all_scores, y_test, title_suffix=""):
    """
    Visualize ONLY supervised models (weighted + SMOTE).
    """
    sup_df = results_df[results_df['Type'] != 'anomaly'].copy()
    if sup_df.empty:
        print("No supervised models to visualize.")
        return

    n_models = len(sup_df)
    fig = plt.figure(figsize=(18, 10))
    gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)

    # 1) Performance bars
    ax1 = fig.add_subplot(gs[0, :])
    x = np.arange(n_models)
    width = 0.22

    ax1.bar(x - 1.5*width, sup_df['Precision'], width, label='Precision', alpha=0.85)
    ax1.bar(x - 0.5*width, sup_df['Recall'], width, label='Recall', alpha=0.85)
    ax1.bar(x + 0.5*width, sup_df['F1-Score'], width, label='F1-Score', alpha=0.85)
    ax1.bar(x + 1.5*width, sup_df['ROC-AUC'], width, label='ROC-AUC', alpha=0.85)
    ax1.set_xticks(x)
    ax1.set_xticklabels(sup_df['Model'], rotation=40, ha='right')
    ax1.set_ylim([0, 1.1])
    ax1.set_title(f"Supervised Models — Performance{title_suffix}")
    ax1.grid(axis='y', alpha=0.3)
    ax1.legend(loc='upper left')

    # 2) Confusion matrix of the best 2 supervised models by F1
    top_models = sup_df.sort_values('F1-Score', ascending=False).head(2)['Model'].tolist()
    for i, m in enumerate(top_models):
        ax = fig.add_subplot(gs[1, i])
        y_pred = all_predictions[m]
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                    xticklabels=['Healthy', 'Leakage'], yticklabels=['Healthy', 'Leakage'], ax=ax)
        ax.set_title(f'{m} — Confusion Matrix')
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')

    # 3) ROC curves (top 5 by F1)
    # put on ax in first row right? we've used both bottom subplots; ROC can go in ax1? We'll add overlay on ax1 is crowded:
    # Instead save fig and break out.
    plt.savefig('supervised_benchmark.png', dpi=300, bbox_inches='tight')
    print("✓ Supervised benchmark saved as 'supervised_benchmark.png'")
    plt.show()

    # Separate ROC figure
    fig2, ax = plt.subplots(figsize=(8, 6))
    for m in sup_df.sort_values('F1-Score', ascending=False).head(5)['Model']:
        scores = all_scores[m]
        try:
            fpr, tpr, _ = roc_curve(y_test, scores)
            auc = roc_auc_score(y_test, scores)
            ax.plot(fpr, tpr, label=f'{m} (AUC={auc:.3f})', linewidth=2)
        except Exception:
            pass
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    ax.set_title(f"Supervised Models — ROC Curves{title_suffix}")
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(alpha=0.3)
    plt.savefig('supervised_roc.png', dpi=300, bbox_inches='tight')
    print("✓ Supervised ROC curves saved as 'supervised_roc.png'")
    plt.show()


def visualize_anomaly_benchmark(results_df, all_predictions, all_scores, y_test, title_suffix=""):
    """
    Visualize ONLY anomaly detection models (IF, OCSVM, LOF).
    Includes: metric bars, score histograms by class, ROC curves.
    """
    ano_df = results_df[results_df['Type'] == 'anomaly'].copy()
    if ano_df.empty:
        print("No anomaly models to visualize.")
        return

    # --- 1) Bars of Precision / Recall / F1 (they might be zero if no positives predicted)
    fig = plt.figure(figsize=(14, 6))
    ax1 = fig.add_subplot(1, 1, 1)
    x = np.arange(len(ano_df)); width = 0.25
    ax1.bar(x - width, ano_df['Precision'], width, label='Precision', alpha=0.9)
    ax1.bar(x,         ano_df['Recall'],    width, label='Recall',    alpha=0.9)
    ax1.bar(x + width, ano_df['F1-Score'],  width, label='F1-Score',  alpha=0.9)
    ax1.set_xticks(x)
    ax1.set_xticklabels(ano_df['Model'], rotation=20, ha='right')
    ax1.set_ylim([0, 1.05])
    ax1.set_title(f"Anomaly Models — Discrete Metrics (current thresholds){title_suffix}")
    ax1.grid(axis='y', alpha=0.3)
    ax1.legend()
    plt.savefig('anomaly_metrics.png', dpi=300, bbox_inches='tight')
    print("✓ Anomaly metrics saved as 'anomaly_metrics.png'")
    plt.show()

    # --- 2) Score histograms by class for each anomaly model
    # Scores are higher = more anomalous (you inverted IsolationForest already)
    for m in ano_df['Model']:
        scores = all_scores[m]
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.hist(scores[y_test == 0], bins=30, alpha=0.6, label='Healthy', density=True)
        ax.hist(scores[y_test == 1], bins=30, alpha=0.6, label='Leakage', density=True)
        ax.set_title(f'{m} — Anomaly Score Distributions{title_suffix}')
        ax.set_xlabel('Anomaly score'); ax.set_ylabel('Density')
        ax.legend()
        ax.grid(alpha=0.3)
        fname = f"{m.replace(' ', '_').lower()}_score_hist.png"
        plt.savefig(fname, dpi=300, bbox_inches='tight')
        print(f"✓ Score histogram saved as '{fname}'")
        plt.show()

    # --- 3) ROC curves for anomaly models (using scores)
    fig, ax = plt.subplots(figsize=(8, 6))
    for m in ano_df['Model']:
        scores = all_scores[m]
        try:
            fpr, tpr, _ = roc_curve(y_test, scores)
            auc = roc_auc_score(y_test, scores)
            ax.plot(fpr, tpr, label=f'{m} (AUC={auc:.3f})', linewidth=2)
        except Exception:
            pass
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    ax.set_title(f"Anomaly Models — ROC Curves{title_suffix}")
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(alpha=0.3)
    plt.savefig('anomaly_roc.png', dpi=300, bbox_inches='tight')
    print("✓ Anomaly ROC curves saved as 'anomaly_roc.png'")
    plt.show()

# ============================================================================
# STEP 8: GENERATE DETAILED REPORT
# ============================================================================

def generate_report(results_df):
    """
    Generate detailed text report
    """
    print("\n" + "="*80)
    print("COMPREHENSIVE MODEL COMPARISON REPORT")
    print("="*80)
    
    print("\n📊 COMPLETE RESULTS TABLE:")
    print("-"*80)
    print(results_df.to_string(index=False))
    
    print("\n" + "="*80)
    print("🏆 TOP 3 MODELS")
    print("="*80)
    
    for idx, row in results_df.head(3).iterrows():
        print(f"\n#{idx+1}: {row['Model']}")
        print(f"  • Type: {row['Type']}")
        print(f"  • Precision: {row['Precision']:.3f} (of predicted leakages, {row['Precision']*100:.1f}% were correct)")
        print(f"  • Recall: {row['Recall']:.3f} (detected {row['Recall']*100:.1f}% of actual leakages)")
        print(f"  • F1-Score: {row['F1-Score']:.3f}")
        print(f"  • ROC-AUC: {row['ROC-AUC']:.3f}")
        print(f"  • True Positives: {row['True Positives']} | False Positives: {row['False Positives']}")
        print(f"  • False Negatives: {row['False Negatives']} | True Negatives: {row['True Negatives']}")
    
    print("\n" + "="*80)
    print("💡 RECOMMENDATIONS")
    print("="*80)
    
    best_model = results_df.iloc[0]
    
    if best_model['Recall'] > 0.8:
        print(f"✓ {best_model['Model']} shows excellent recall ({best_model['Recall']:.1%})")
        print("  → Good at catching leakages, recommended for safety-critical applications")
    elif best_model['Precision'] > 0.8:
        print(f"✓ {best_model['Model']} shows excellent precision ({best_model['Precision']:.1%})")
        print("  → Low false alarms, recommended for cost-sensitive applications")
    
    if best_model['F1-Score'] > 0.7:
        print(f"✓ Strong overall performance (F1={best_model['F1-Score']:.3f})")
    else:
        print("⚠ Consider collecting more leakage samples to improve model performance")
    
    print("\n" + "="*80)

def generate_grouped_report(results_df):
    print("\n" + "="*80)
    print("COMPREHENSIVE MODEL COMPARISON REPORT (Grouped)")
    print("="*80)

    if not results_df[results_df['Type'] != 'anomaly'].empty:
        print("\n--- SUPERVISED MODELS ---")
        print(results_df[results_df['Type'] != 'anomaly'].sort_values('F1-Score', ascending=False).to_string(index=False))

    if not results_df[results_df['Type'] == 'anomaly'].empty:
        print("\n--- ANOMALY MODELS ---")
        print(results_df[results_df['Type'] == 'anomaly'].sort_values('F1-Score', ascending=False).to_string(index=False))

# ============================================================================
# MAIN PIPELINE
# ============================================================================

def main():
    """
    Complete pipeline execution with multiple models
    """
    print("="*80)
    print("MULTI-MODEL ANOMALY DETECTION PIPELINE")
    print("="*80)
    
    # Step 1: Load data
    print("\n[1/7] Loading data...")
    df = load_data("model.csv", "TestBrakefinal_data_healthy.csv")  # Replace with your data loading
    
    # Step 2: Preprocess
    print("\n[2/7] Preprocessing data...")
    X_train, X_test, y_train, y_test, X_train_healthy = preprocess_data(df)
    
    # Step 3: Impute + Scale features
    print("\n[3/7] Imputing (median) and scaling features...")
    X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer = scale_features(
    X_train, X_test, X_train_healthy
    )
    
    # Step 4: Train all models
    print("\n[4/7] Training all models...")
    contamination_rate = 23 / 1123
    trained_models = train_all_models(
        X_train_scaled, y_train, X_train_healthy_scaled, contamination_rate
    )
    
    # Step 5: Evaluate all models
    print("\n[5/7] Evaluating all models...")
    results_df, all_predictions, all_scores = evaluate_all_models(
        trained_models, X_test_scaled, y_test
    )
    
    # Step 6: Visualize comparison (separate)
    print("\n[6/7] Creating separate benchmarks...")
    visualize_supervised_benchmark(results_df, all_predictions, all_scores, y_test, title_suffix="")
    visualize_anomaly_benchmark(results_df, all_predictions, all_scores, y_test, title_suffix="")

    # Step 7: Generate report
    print("\n[7/7] Generating detailed report...")
    generate_report(results_df)
    

    # Step 8: Generate grouped report
    print("\n[7/7] Generating grouped report...")
    generate_grouped_report(results_df)
        
    print("\n" + "="*80)
    print("✓ PIPELINE COMPLETE!")
    print("="*80)
    
    return trained_models, results_df, scaler

# ============================================================================
# RUN PIPELINE
# ============================================================================

if __name__ == "__main__":
    trained_models, results_df, scaler = main()
    
    # Access best model
    best_model_name = results_df.iloc[0]['Model']
    best_model = trained_models[best_model_name]['model']
    
    print(f"\n🎯 Best Model: {best_model_name}")
    print(f"   F1-Score: {results_df.iloc[0]['F1-Score']:.3f}")
    
    # Save best model (optional)
    # import joblib
    # joblib.dump(best_model, f'{best_model_name.replace(" ", "_")}_model.pkl')
    # joblib.dump(scaler, 'scaler.pkl')
    # joblib.dump(imputer, 'imputer.pkl')